<a href="https://colab.research.google.com/github/trimpouo/Neuromorphic_ReID/blob/main/EPRBench/colab_train_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EPRBench: Event Stream Based Visual Place Recognition (SGVPR)

本 Notebook 用于在 **Google Colab** 中直接运行 `Neuromorphic_ReID/EPRBench`（论文 *EPRBench: A High-Quality Benchmark Dataset for Event Stream Based Visual Place Recognition*，模型 **SGVPR**），支持：

1. **GPU 环境检查与依赖安装**：一键配置 `faiss`、`open_clip_torch`、`pytorch-metric-learning`、`timm` 与 `7z` 解压工具。
2. **代码仓库准备**：自动拉取或定位 `Neuromorphic_ReID/EPRBench` 工作目录，并处理 Colab 高版本 PyTorch / NumPy 兼容性。
3. **挂载 Google Drive 与数据集高速解压**：自动挂载云盘 `/content/drive/MyDrive/datasets/EPRBench`，校验 `EventVPR.7z`、`dinov2_vitb14_pretrain.pth` 与 `SGVPR.pth`，并将数据集解压至 Colab 本地高速磁盘 `/content/datasets/EventVPR`。
4. **预训练模型评估 (`eval.py`)**：直接加载云盘中的 `SGVPR.pth` 在测试集上计算 `R@1 / R@5 / R@10 / R@100`。
5. **模型训练 (`train.py`) 与结果备份**：加载 `dinov2_vitb14_pretrain.pth` 从头训练 SGVPR，并支持将训练好的权重与日志自动备份回 Google Drive。

## 1. 检查 GPU 环境与安装依赖
请确保 Colab 运行时已设置为 GPU（`Runtime` -> `Change runtime type` -> `Hardware accelerator: GPU`）。

In [ ]:
!nvidia-smi
!apt-get update -qq && apt-get install -qq -y p7zip-full
!pip install -q faiss-cpu open_clip_torch pytorch-metric-learning timm einops ftfy regex tqdm scikit-learn

## 2. 克隆 GitHub 仓库并进入 `EPRBench` 目录
- 如果你从自己 Fork 的仓库打开，可将下方 `REPO_URL` 改为你自己的仓库地址。
- 如果当前目录已在 `EPRBench` 内，脚本会自动识别并跳过克隆。

In [ ]:
import os
from pathlib import Path

# ================= 配置区 =================
REPO_URL = "https://github.com/trimpouo/Neuromorphic_ReID.git"
FALLBACK_REPO_URL = "https://github.com/wangxiao5791509/Neuromorphic_ReID.git"
REPO_DIR = "/content/Neuromorphic_ReID"
# ==========================================

cwd = Path.cwd()
if (cwd / "train.py").exists() and (cwd / "eval.py").exists() and (cwd / "network.py").exists():
    PROJECT_DIR = str(cwd)
    print(f"当前已在 EPRBench 目录中: {PROJECT_DIR}")
elif (cwd / "EPRBench" / "train.py").exists():
    PROJECT_DIR = str(cwd / "EPRBench")
    os.chdir(PROJECT_DIR)
    print(f"已切换工作目录至: {PROJECT_DIR}")
else:
    if not os.path.exists(REPO_DIR):
        print(f"正在克隆仓库: {REPO_URL} ...")
        ret = os.system(f"git clone {REPO_URL} {REPO_DIR}")
        if ret != 0:
            print(f"主仓库克隆失败，尝试官方仓库: {FALLBACK_REPO_URL} ...")
            !git clone {FALLBACK_REPO_URL} {REPO_DIR}
    PROJECT_DIR = os.path.join(REPO_DIR, "EPRBench")
    os.chdir(PROJECT_DIR)
    print(f"已切换工作目录至: {os.getcwd()}")

# 兼容 Colab 上的 PyTorch 2.6+ (默认 weights_only=True 会拦截含 numpy 对象的 checkpoint) 及 NumPy 2.x
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
os.environ["PYTHONHASHSEED"] = "0"

test_py = Path("test.py")
if test_py.exists():
    content = test_py.read_text(encoding="utf-8")
    if "np.in1d(" in content:
        test_py.write_text(content.replace("np.in1d(", "np.isin("), encoding="utf-8")
        print("已自动更新 test.py 中的 np.in1d -> np.isin 以兼容 NumPy 2.x")

!ls -la

## 3. 挂载 Google Drive 并校验云盘数据与模型文件
根据云盘目录结构，默认路径为 `/content/drive/MyDrive/datasets/EPRBench`：
- 数据集压缩包：`/content/drive/MyDrive/datasets/EPRBench/EventVPR.7z` (约 13.2 GB)
- DINOv2 预训练底座：`/content/drive/MyDrive/datasets/EPRBench/model/dinov2_vitb14_pretrain.pth` (约 330.3 MB)
- SGVPR 训练好权重：`/content/drive/MyDrive/datasets/EPRBench/model/SGVPR.pth` (约 1.67 GB)

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

# ================= 云盘路径配置 =================
GDRIVE_EPRBENCH_DIR = "/content/drive/MyDrive/datasets/EPRBench"
GDRIVE_ARCHIVE_PATH = os.path.join(GDRIVE_EPRBENCH_DIR, "EventVPR.7z")
DINOV2_MODEL_PATH   = os.path.join(GDRIVE_EPRBENCH_DIR, "model", "dinov2_vitb14_pretrain.pth")
SGVPR_MODEL_PATH    = os.path.join(GDRIVE_EPRBENCH_DIR, "model", "SGVPR.pth")
# ================================================

for name, path in [
    ("EventVPR 数据集压缩包", GDRIVE_ARCHIVE_PATH),
    ("DINOv2 预训练权重", DINOV2_MODEL_PATH),
    ("SGVPR 预训练权重", SGVPR_MODEL_PATH),
]:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"✅ {name}: {path} ({size_mb:.1f} MB)")
    else:
        print(f"❌ 未找到 {name}: {path}，请检查云盘路径！")

## 4. 解压 `EventVPR.7z` 到 Colab 本地高速磁盘
- **重要提示**：切勿将包含数万张小图的 `EventVPR.7z` 解压回 Google Drive，否则会导致极慢的 I/O 甚至触发云盘读写限流。
- 下方脚本直接从 Google Drive 流式读取 `EventVPR.7z` 并解压到 Colab 本地 SSD (`/content/datasets/EventVPR`)，既节省 13.2 GB 本地压缩包拷贝空间，又能保证训练/测试时的高速数据加载。
- 注意：`eval.py` 内部通过 `join(eval_datasets_folder, eval_dataset_name)` 拼接路径，因此：
  - `LOCAL_DATASETS_ROOT = "/content/datasets"`（传给 `eval.py` 的 `--eval_datasets_folder`）
  - `DATASET_DIR = "/content/datasets/EventVPR"`（传给 `train.py` 的 `--dataset_folder`）

In [ ]:
import os
import shutil
from pathlib import Path

LOCAL_DATASETS_ROOT = "/content/datasets"
DATASET_DIR = os.path.join(LOCAL_DATASETS_ROOT, "EventVPR")
TEXT_FOLDER = os.path.join(DATASET_DIR, "scene_descriptions")

os.makedirs(LOCAL_DATASETS_ROOT, exist_ok=True)

if os.path.exists(os.path.join(DATASET_DIR, "train")) and os.path.exists(os.path.join(DATASET_DIR, "test")):
    print(f"✅ 数据集已就绪，跳过解压: {DATASET_DIR}")
else:
    assert os.path.exists(GDRIVE_ARCHIVE_PATH), f"找不到压缩包: {GDRIVE_ARCHIVE_PATH}"
    print(f"正在将 {GDRIVE_ARCHIVE_PATH} 解压至本地磁盘 {LOCAL_DATASETS_ROOT} (约需数分钟，请耐心等待)...")
    !7z x "{GDRIVE_ARCHIVE_PATH}" -o"{LOCAL_DATASETS_ROOT}" -y > /dev/null
    print("解压完成，正在整理目录结构...")

    # 自动探测解压后的实际目录位置并规范化至 /content/datasets/EventVPR
    if not (os.path.exists(os.path.join(DATASET_DIR, "train")) and os.path.exists(os.path.join(DATASET_DIR, "test"))):
        # 情况 1: train/test 直接解压到了 /content/datasets/ 下
        if os.path.exists(os.path.join(LOCAL_DATASETS_ROOT, "train")) and os.path.exists(os.path.join(LOCAL_DATASETS_ROOT, "test")):
            os.makedirs(DATASET_DIR, exist_ok=True)
            for sub in ["train", "val", "test", "scene_descriptions"]:
                src = os.path.join(LOCAL_DATASETS_ROOT, sub)
                if os.path.exists(src):
                    shutil.move(src, os.path.join(DATASET_DIR, sub))
        else:
            # 情况 2: 嵌套在深层目录中
            for root, dirs, _ in os.walk(LOCAL_DATASETS_ROOT):
                if "train" in dirs and "test" in dirs and Path(root) != Path(DATASET_DIR):
                    print(f"检测到数据集位于 {root}，正在移动至 {DATASET_DIR} ...")
                    if os.path.exists(DATASET_DIR):
                        shutil.rmtree(DATASET_DIR)
                    shutil.move(root, DATASET_DIR)
                    break

# 校验数据集各子目录完整性
for split in ["train", "val", "test"]:
    split_dir = Path(DATASET_DIR) / split
    if split_dir.exists():
        db_scenes = len(list((split_dir / "database").glob("@*"))) if (split_dir / "database").exists() else 0
        q_scenes = len(list((split_dir / "queries").glob("@*"))) if (split_dir / "queries").exists() else 0
        print(f"  - {split:5s}: database 场景数 = {db_scenes}, queries 场景数 = {q_scenes}")

txt_count = len(list(Path(TEXT_FOLDER).glob("*.txt"))) if os.path.exists(TEXT_FOLDER) else 0
print(f"  - scene_descriptions 文本文件数 = {txt_count}")

## 5. 测试预训练模型 (`eval.py`)
直接使用云盘中的 `SGVPR.pth` 在 `EventVPR` 测试集上进行评估，输出 `R@1, R@5, R@10, R@100`。

In [ ]:
!TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 python eval.py \
    --eval_datasets_folder "{LOCAL_DATASETS_ROOT}" \
    --eval_dataset_name EventVPR \
    --text_folder "{TEXT_FOLDER}" \
    --resume "{SGVPR_MODEL_PATH}" \
    --infer_batch_size 16 \
    --num_workers 2 \
    --use_text

## 6. 训练模型 (`train.py`)
使用 `dinov2_vitb14_pretrain.pth` 作为视觉底座初始化模型并开始训练。
- **显存自适应调整**：`EventVPRDataset` 每个训练样本包含 `img_per_place = 5` 张图像，实际前向传播的图像数为 `train_batch_size * 5`。
  - 若分配到 **L4 (24GB) / A100 (40GB/80GB)**，将自动使用官方默认的 `train_batch_size = 24`。
  - 若分配到 **T4 (15GB)**，将自动下调至 `train_batch_size = 8` 以防止显存溢出 (OOM)。
- 训练日志与权重保存在 `logs/sgvpr_colab/<时间戳>/` 下（包含 `best_model.pth` 和 `last_model.pth`）。

In [ ]:
import torch

gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3) if torch.cuda.is_available() else 0
if gpu_mem_gb >= 22:
    TRAIN_BATCH_SIZE = 24
elif gpu_mem_gb >= 14:
    TRAIN_BATCH_SIZE = 8
else:
    TRAIN_BATCH_SIZE = 4

EPOCHS_NUM = 50
LR = 0.0001
SAVE_DIR = "sgvpr_colab"

print(f"检测到 GPU 显存: {gpu_mem_gb:.1f} GB -> 自动设置 train_batch_size = {TRAIN_BATCH_SIZE}")

!TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 python train.py \
    --dataset_folder "{DATASET_DIR}" \
    --foundation_model_path "{DINOV2_MODEL_PATH}" \
    --text_folder "{TEXT_FOLDER}" \
    --save_dir "{SAVE_DIR}" \
    --train_batch_size {TRAIN_BATCH_SIZE} \
    --infer_batch_size 16 \
    --epochs_num {EPOCHS_NUM} \
    --lr {LR} \
    --lambda_contrast 0.25 \
    --temperature 0.07 \
    --num_workers 2 \
    --use_text

## 7. 评估新训练的模型 & 备份权重到 Google Drive
训练完成后，可运行下方单元格测试最新保存的 `best_model.pth`，并将训练日志和模型权重备份到 Google Drive 的 `/content/drive/MyDrive/datasets/EPRBench/logs_backup` 目录中，防止 Colab 断开后丢失。

In [ ]:
import glob
import os
import shutil

best_ckpts = sorted(glob.glob(os.path.join(PROJECT_DIR, "logs", "sgvpr_colab", "*", "best_model.pth")))
if best_ckpts:
    LATEST_BEST_MODEL = best_ckpts[-1]
    print(f"找到最新训练的最佳模型: {LATEST_BEST_MODEL}")
    !TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 python eval.py \
        --eval_datasets_folder "{LOCAL_DATASETS_ROOT}" \
        --eval_dataset_name EventVPR \
        --text_folder "{TEXT_FOLDER}" \
        --resume "{LATEST_BEST_MODEL}" \
        --infer_batch_size 16 \
        --num_workers 2 \
        --use_text
else:
    print("未找到 logs/sgvpr_colab/*/best_model.pth，请先运行训练单元格。")

# 是否将训练日志和权重备份回 Google Drive
BACKUP_TO_DRIVE = True

if BACKUP_TO_DRIVE and os.path.exists(os.path.join(PROJECT_DIR, "logs")):
    backup_dir = os.path.join(GDRIVE_EPRBENCH_DIR, "logs_backup")
    shutil.copytree(os.path.join(PROJECT_DIR, "logs"), backup_dir, dirs_exist_ok=True)
    print(f"✅ 已将训练日志与模型权重备份至 Google Drive: {backup_dir}")